In [54]:
#Step 1: Install the required libraries

!pip install requests
!pip install datasets

In [55]:
#Step 2: Import the required libraries

from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path
 
# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
 
# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

Loading product dataset...
✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


In [56]:
print(products_df.columns.tolist())

['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']


In [57]:
from pathlib import Path

In [61]:
# Step 2 (continued): Add placeholder prices
# The dataset has no price column, so each main category gets a placeholder price These prices are NOT real; they only make the prompt template work

placeholder_prices = {
    "Apparel": 29.99,
    "Accessories": 24.99,
    "Footwear": 59.99,
    "Personal Care": 14.99,
}

# Look up the price by masterCategory; any other category gets 19.99
products_df['price'] = products_df['masterCategory'].map(placeholder_prices).fillna(19.99)

print("✓ Placeholder prices added")
print(products_df[['productDisplayName', 'masterCategory', 'price']].head(10))

✓ Placeholder prices added
                              productDisplayName masterCategory  price
0               Turtle Check Men Navy Blue Shirt        Apparel  29.99
1             Peter England Men Party Blue Jeans        Apparel  29.99
2                       Titan Women Silver Watch    Accessories  24.99
3  Manchester United Men Solid Black Track Pants        Apparel  29.99
4                          Puma Men Grey T-shirt        Apparel  29.99
5           Inkfruit Mens Chain Reaction T-shirt        Apparel  29.99
6               Fabindia Men Striped Green Shirt        Apparel  29.99
7                  Jealous 21 Women Purple Shirt        Apparel  29.99
8                       Puma Men Pack of 3 Socks    Accessories  24.99
9                         Skagen Men Black Watch    Accessories  24.99


In [62]:
# Step 2 (continued): Organize product images

# Save each product image to the product_images folder and store its file path in a new 'image_path' column
image_paths = []
for index, row in products_df.iterrows():
    path = images_dir / f"{row['id']}.jpg"
    row['image'].convert("RGB").save(path)
    image_paths.append(str(path))

products_df['image_path'] = image_paths

print(f"✓ Saved {len(image_paths)} images to '{images_dir}'")
print(products_df[['id', 'productDisplayName', 'image_path']].head())

✓ Saved 100 images to 'product_images'
      id                             productDisplayName  \
0  15970               Turtle Check Men Navy Blue Shirt   
1  39386             Peter England Men Party Blue Jeans   
2  59263                       Titan Women Silver Watch   
3  21379  Manchester United Men Solid Black Track Pants   
4  53759                          Puma Men Grey T-shirt   

                 image_path  
0  product_images/15970.jpg  
1  product_images/39386.jpg  
2  product_images/59263.jpg  
3  product_images/21379.jpg  
4  product_images/53759.jpg  


In [64]:
#Step 3: Encode images to base64

import base64

def encode_image_to_base64(image_path):
    """Encode an image file to base64 string."""
    with open(image_path, "rb") as img_file:
        encoded = base64.b64encode(img_file.read()).decode("utf-8")
    return encoded
# Example usage
sample_path = products_df.iloc[0]["image_path"]
encoded_image = encode_image_to_base64(sample_path)
print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...") 



Encoded image length: 2388 characters
Encoded prefix: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcG...


In [65]:
#Step 4: Create a prompt template for product listing generation

def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.
 
Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt
 
# Test prompt creation
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)
 
print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt[:500] + "...")  # Show first 500 characters


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive ...


In [66]:
#Step 5: Calling the ChatGPT API with Vision

# Install the OpenAI library and the .env loader (only needed once)
!pip install openai python-dotenv

from openai import OpenAI
from dotenv import load_dotenv
import json

# Read the API key from the .env file, so it never appears in the notebook
load_dotenv()

# The client picks up OPENAI_API_KEY automatically
client = OpenAI()
print("✓ OpenAI client ready")

✓ OpenAI client ready


In [ ]:
#Step 5: Prepare the API request with image and prompt

def prepare_api_request(prompt, encoded_image):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{encoded_image}"}
                }
            ]
        }
    ]
    return messages
    
# Test with the first product from the dataset
sample = products_df.iloc[0]

sample_prompt = create_product_listing_prompt(
    product_name=sample['productDisplayName'],
    price=sample['price'],
    category=sample['masterCategory'],
    additional_info=f"{sample['baseColour']}, {sample['gender']}, {sample['usage']}"
)

sample_image = encode_image_to_base64(sample['image_path'])
sample_messages = prepare_api_request(sample_prompt, sample_image)

print(f"Product: {sample['productDisplayName']}")
print(f"Prompt length: {len(sample_prompt)} characters")
print(f"Image length: {len(sample_image)} characters")
print("✓ API request prepared")

Product: Turtle Check Men Navy Blue Shirt
Prompt length: 1011 characters
Image length: 2388 characters
✓ API request prepared


In [68]:
#Step 5 (continued): Call the ChatGPT API

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=sample_messages
)

print("✓ Response received")

✓ Response received


In [69]:
#Step 5 (continued): Handle the response

# Take the text out of the response
listing_text = response.choices[0].message.content

# The model sometimes wraps JSON in ```json ... ``` markers, so remove them
listing_text = listing_text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

print("=" * 50)
print("RAW RESPONSE")
print("=" * 50)
print(listing_text)

RAW RESPONSE
{
    "title": "Stylish Turtle Check Navy Blue Shirt for Men",
    "description": "Elevate your casual wardrobe with our Turtle Check Men Navy Blue Shirt, designed for the modern man who values style and comfort. Crafted from high-quality, breathable fabric, this shirt combines a sophisticated navy blue hue with an eye-catching checkered pattern, making it perfect for both relaxed weekends and casual outings. The short sleeves allow for easy movement, ensuring you stay cool on warm days. With a classic cut that flatters all body types and attention to detail in stitching and finishes, this shirt ensures you look polished yet approachable. Pair it effortlessly with jeans or chinos for a complete look that’s effortlessly stylish. Available now for just $29.99, this shirt is a must-have addition to your wardrobe.",
    "features": [
        "Vibrant navy blue color with a classic checkered pattern",
        "Short sleeves for a comfortable fit",
        "Breathable fabric ide

In [70]:
#Step 5 (continued): Parse JSON output

try:
    listing = json.loads(listing_text)

    print("=" * 50)
    print("PRODUCT LISTING")
    print("=" * 50)
    print(f"Title: {listing['title']}")
    print(f"\nDescription: {listing['description']}")
    print("\nFeatures:")
    for feature in listing['features']:
        print(f"  - {feature}")
    print(f"\nKeywords: {listing['keywords']}")

except json.JSONDecodeError:
    print("⚠ The response is not valid JSON. Check the raw response above.")

PRODUCT LISTING
Title: Stylish Turtle Check Navy Blue Shirt for Men

Description: Elevate your casual wardrobe with our Turtle Check Men Navy Blue Shirt, designed for the modern man who values style and comfort. Crafted from high-quality, breathable fabric, this shirt combines a sophisticated navy blue hue with an eye-catching checkered pattern, making it perfect for both relaxed weekends and casual outings. The short sleeves allow for easy movement, ensuring you stay cool on warm days. With a classic cut that flatters all body types and attention to detail in stitching and finishes, this shirt ensures you look polished yet approachable. Pair it effortlessly with jeans or chinos for a complete look that’s effortlessly stylish. Available now for just $29.99, this shirt is a must-have addition to your wardrobe.

Features:
  - Vibrant navy blue color with a classic checkered pattern
  - Short sleeves for a comfortable fit
  - Breathable fabric ideal for casual wear
  - Tailored cut to enh

In [71]:
#Step 6: Processing Multiple Products

# Generate a listing for one product, reusing the Step 5 code
def generate_listing(product):
    prompt = create_product_listing_prompt(
        product_name=product['productDisplayName'],
        price=product['price'],
        category=product['masterCategory'],
        additional_info=f"{product['baseColour']}, {product['gender']}, {product['usage']}"
    )
    encoded_image = encode_image_to_base64(product['image_path'])
    messages = prepare_api_request(prompt, encoded_image)

    # Call the API (same as Step 5)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    # Handle the response (same as Step 5)
    listing_text = response.choices[0].message.content
    listing_text = listing_text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

    # Parse the JSON (same as Step 5)
    return json.loads(listing_text)


# Print a listing in the same format as Step 5
def print_listing(listing):
    print(f"Title: {listing['title']}")
    print(f"\nDescription: {listing['description']}")
    print("\nFeatures:")
    for feature in listing['features']:
        print(f"  - {feature}")
    print(f"\nKeywords: {listing['keywords']}")

print("✓ Functions ready")

✓ Functions ready


In [73]:
#Step 6.(continued): Generate  listing for products, handle errors

NUM_PRODUCTS = 20   # First 20 products, to keep API cost low
results = []
errors = []

for index, product in products_df.head(NUM_PRODUCTS).iterrows():
    print("=" * 50)
    print(f"PRODUCT {index + 1}: {product['productDisplayName']}")
    print("=" * 50)

    try:
        listing = generate_listing(product)
        print_listing(listing)

        results.append({
            "id": int(product['id']),
            "product_name": product['productDisplayName'],
            "category": product['masterCategory'],
            "price": float(product['price']),
            "listing": listing
        })

    except Exception as e:
        # One failed product does not stop the loop
        print(f"⚠ Could not generate listing: {e}")
        errors.append({"id": int(product['id']), "product_name": product['productDisplayName'], "error": str(e)})

    print()

print(f"✓ Done: {len(results)} listings generated, {len(errors)} failed")

PRODUCT 1: Turtle Check Men Navy Blue Shirt
Title: Stylish Navy Blue Turtle Check Shirt for Men

Description: Elevate your casual style with our Turtle Check Men Navy Blue Shirt. Made from high-quality, breathable fabric, this shirt ensures all-day comfort while keeping you looking sharp. The subtle navy blue color is perfect for any occasion, whether you're heading to brunch or a night out with friends. The classic check pattern adds a touch of modern sophistication, while the short sleeves make it ideal for warmer days. Pair it effortlessly with your favorite jeans or shorts for a laid-back yet stylish look. Don’t miss out on adding this versatile piece to your wardrobe!

Features:
  - Stylish and modern navy blue check pattern
  - Lightweight, breathable fabric for all-day comfort
  - Short sleeves for a relaxed, casual feel
  - Versatile style perfect for various occasions
  - Easy to pair with jeans, chinos, or shorts
  - Machine washable for easy care
  - Available at an affordab

In [74]:
#Step 6 (continued): Save results

with open("generated_listings.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✓ Saved {len(results)} listings to 'generated_listings.json'")

if errors:
    print(f"\n⚠ {len(errors)} products failed:")
    for error in errors:
        print(f"  - {error['product_name']}: {error['error']}")

✓ Saved 20 listings to 'generated_listings.json'


## Summary

### What this notebook does
It generates e-commerce product listings automatically. For each product, the notebook sends the product photo and its basic details to the ChatGPT API (model `gpt-4o-mini`), which returns a listing with a title, description, key features and SEO keywords in JSON format.

### Security
The OpenAI API key is stored in a separate `.env` file and is never shown in the notebook. And it is protected by .gitignore